# 03 — Random Forest

**Purpose:** Train the supervised multiclass classifier (5 classes).  
**Acceptance criterion:** macro F1 ≥ 0.80 on ≥ 3 of 4 attack categories, measured on **val** set.  
If criterion not met, iterate on features (return to extractor tasks 1.x) before touching model.

**Input:** `training/splits/train.parquet`, `training/splits/val.parquet`  
**Output:** `training/models/rf_vN.pkl`, `training/results/rf_val_report.txt`

**Rule R2:** Hyperparameter tuning uses val only. Test set is locked until notebook 05.

## 1. Load Features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

SPLITS  = Path("../../training/splits")
RESULTS = Path("../../training/results")
MODELS  = Path("../../training/models")
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(SPLITS / "train.parquet")
val_df   = pd.read_parquet(SPLITS / "val.parquet")

DROP_COLS = [
    "status_code", "req_count_1s", "req_count_5s", "req_count_60s",
    "error_rate_4xx_60s", "endpoint_diversity_60s", "_source",
    "sample_id", "timestamp",
]
train_df.drop(columns=[c for c in DROP_COLS if c in train_df.columns], inplace=True)
val_df.drop(columns=[c for c in DROP_COLS if c in val_df.columns], inplace=True)

y_train = train_df.pop("label")
y_val   = val_df.pop("label")
X_train, X_val = train_df, val_df

TARGET_NAMES = ["benign", "cmdi", "path_traversal", "sqli", "xss"]

print(f"Train: {X_train.shape}  Val: {X_val.shape}")
print(y_train.value_counts())

## 2. Hyperparameter Search

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators":      [100, 200, 300],
    "max_depth":         [None, 20, 40],
    "min_samples_split": [2, 5, 10],
    "class_weight":      ["balanced", "balanced_subsample"],
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

search = RandomizedSearchCV(
    rf_base,
    param_dist,
    n_iter=20,
    cv=5,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
    verbose=2,
    refit=True,
)
search.fit(X_train, y_train)

print(f"Best params : {search.best_params_}")
print(f"Best CV F1  : {search.best_score_:.4f}")
best_rf = search.best_estimator_

## 3. Best Model Evaluation on Val

In [ ]:
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred_rf = best_rf.predict(X_val)

print("=== Random Forest — Validation Set ===")
print(classification_report(y_val, y_pred_rf, target_names=TARGET_NAMES))

macro_f1  = f1_score(y_val, y_pred_rf, average="macro")
per_class = f1_score(y_val, y_pred_rf, average=None,
                     labels=["cmdi", "path_traversal", "sqli", "xss"])

print(f"Macro F1           : {macro_f1:.4f}")
print(f"Attack classes >= 0.80: {sum(f >= 0.80 for f in per_class)}/4")

# Gate check — must pass before proceeding to notebook 04
assert sum(f >= 0.80 for f in per_class) >= 3, (
    "GATE FAILED: F1 >= 0.80 in fewer than 3 attack classes. "
    "Return to feature engineering (tasks 1.x) before proceeding."
)

## 4. Feature Importance

In [ ]:
importances = pd.Series(
    best_rf.feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)

print("Top 20 features (mean impurity decrease):")
print(importances.head(20).to_string())

low_importance = importances[importances < 0.001].index.tolist()
print(f"\nFeatures with importance < 0.001 : {len(low_importance)}")
print("Candidates for pruning:", low_importance)

# Permutation importance (more reliable — use on val, not train)
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_rf, X_val, y_val,
    n_repeats=10, random_state=42, n_jobs=-1, scoring="f1_macro",
)
perm_series = pd.Series(
    perm.importances_mean, index=X_train.columns
).sort_values(ascending=False)

print("\nTop 20 features (permutation importance on val):")
print(perm_series.head(20).to_string())

## 5. Save Model

In [ ]:
import joblib

model_path = MODELS / "rf_v1.pkl"
joblib.dump(best_rf, model_path)
print(f"Model saved : {model_path}")
print(f"Params      : {best_rf.get_params()}")

## 6. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred_rf,
    display_labels=TARGET_NAMES,
    ax=ax,
    colorbar=False,
    xticks_rotation=30,
)
plt.title("Random Forest — Confusion Matrix (val set)")
plt.tight_layout()

fig_path = RESULTS / "rf_confusion_matrix.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved to {fig_path}")